# 01. Inventário do banco do Lab para a linha MCMV-Entidades (FDS)

Banco `10.0.0.50/cidades`: Postgres medalhão do Gov Hub (schemas `bronze`, `prata`, `ouro`, `seeds`), materializado pelo dbt do `data-application-cidades` (pacote `fds_dbt`, mais o eixo histórico `mcmv_historico_dbt`). O Entidades é a linha de autogestão: quem propõe, contrata e executa é uma Entidade Organizadora (EO), com as famílias organizadas em comissões (CRE e CAO). O dinheiro sai da LOA pela ação 00CY, entra no Fundo de Desenvolvimento Social (FDS) e a CAIXA opera.

Perguntas deste notebook: (1) que tabelas descrevem a carteira do Novo MCMV-Entidades e as entidades, (2) onde está a série no tempo (físico, financeiro, situação) e o histórico do legado PMCMV-E, (3) o que o FAR tem e o Entidades ainda não tem no banco. É o mesmo roteiro do inventário do FAR (`notebooks/far/01_inventario_replica.ipynb`), com a diferença de que aqui a entidade é uma dimensão própria.

<div style="border:2px solid #d03b3b;background:#fff4f4;padding:10px 14px;border-radius:6px;margin:6px 0"><b style="color:#d03b3b">⚠ Fora do banco: resumo da auditoria deste notebook</b><ul style="margin:6px 0 0 0"><li>Todos os números deste notebook (contagens, datas, valores) vêm do Postgres do Lab, schemas prata, ouro e seeds; a bronze entra só na contagem de linhas e no retrato de trabalho social da CAIXA (seção 8), que ainda não tem prata.</li><li>Não são dados do banco: a descrição dos arquivos de origem e dos canais (SharePoint da SNH e SFTP da CAIXA), lida dos modelos dbt, e o contexto da política (portarias, ciclos), que vem do estudo do fluxo e de pesquisa. Esses não precisam de marcação.</li></ul></div>

In [1]:
import sys; sys.path.insert(0, "../..")
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
from src import db
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_columns", 60)
INTERIM = Path("../../data/interim"); INTERIM.mkdir(parents=True, exist_ok=True)
def cache(nome, sql):
    f = INTERIM / f"{nome}.parquet"
    if f.exists(): return pd.read_parquet(f)
    df = db.ler_sql(sql); df.to_parquet(f); return df
print(db.HOST)
db.schemas()

10.0.0.50


,schema,objetos
0,abecip,1
1,bronze,166
2,conjuntura,0
3,duckdb,2
4,fgv,1
5,ibge,14
6,lake,8
7,metadata,1
8,ouro,172
9,prata,141


## 1. As tabelas da linha Entidades por schema

Tudo que tem `fds`, `entidade` ou `int059` no nome. As prata `prata_fds_*` são a tipagem das extrações mensais do GFAR (cadastro PJ, obra, financeiro, trabalho social) que chegam pelo SharePoint da SNH, mais o INT059 e os Dados Prioritários da CAIXA que chegam pelo SFTP. A `prata_hist_fds_empreendimento` é o eixo histórico (retratos mensais de três fontes). As ouro são a ficha do empreendimento e o panorama por entidade que o Superset lê.

In [2]:
tabs = cache("fds_inventario_lista", """
select table_schema as schema, table_name as tabela from information_schema.tables
where table_schema in ('bronze','prata','ouro','seeds') and table_name ~ 'fds|entidade|int059|int_059' order by 1, 2""")
sql = " union all ".join(f"select '{s}' schema, '{t}' tabela, count(*) linhas from {s}.{t}" for s, t in zip(tabs.schema, tabs.tabela))
inv = cache("fds_inventario_linhas", sql).sort_values(["schema", "tabela"])
DESC = {
 "prata_fds_cadastro_pj": "cadastro do empreendimento contratado pela EO (MONIT_CAD_PJ_FDS_MENSAL): composição financeira, tipologia, datas; 1 linha por APF",
 "prata_fds_obra_mensal": "medição física do mês mais recente (MONIT_MOV_OBRA_FDS_MENSAL): situação, % previsto e realizado, UH alienadas; retrato, não série",
 "prata_fds_financeiro_mensal": "liberações do FDS, 1 linha por liberação (MONIT_MOV_FINANC_FDS_MENSAL); a única fonte do GFAR com série",
 "prata_fds_trabalho_social": "execução do trabalho social por APF (MONIT_EXEC_TS_FDS_MENSAL); só quem tem TS reportado",
 "prata_fds_int_059_caixa_pj": "INT059 da CAIXA: situação GEFUS, fase do contrato (projeto ou obra), legado e Novo MCMV",
 "prata_fds_dados_prioritarios_entregas": "entregas da CAIXA (Dados Prioritários), todos os programas; o recorte FDS é no join",
 "prata_fds_empreendimento": "visão unificada por APF: cadastro + obra + INT059 + TS + desembolso acumulado; base da ficha",
 "prata_fds_evolucao_financeira": "desembolso por APF e mês, acumulado e % executado",
 "prata_fds_dim_empreendimento": "identidade do empreendimento: liga APFs de fases distintas (Projeto, Obra, Desligamento) a um id estável",
 "prata_hist_fds_empreendimento": "eixo histórico: retratos mensais do INT059 (desde 2019-12), dos Dados Prioritários (2024-06 a 2026-03) e da obra mensal (2026)",
 "ouro_fds_ficha_empreendimento": "ficha por APF com semáforo, ritmo físico x financeiro e status de entrega (Superset)",
 "ouro_fds_panorama_entidade": "painel por CNPJ de EO: carteira, execução, entregas, score de risco",
 "ouro_fds_evolucao_financeira_chart": "série APF x mês para o gráfico financeiro",
 "seed_apf_fase_fds": "mapeamento curado APF -> fase (Projeto, Obra, Desligamento) e APF âncora, do xlsx GEFUS",
 "bronze_shpt_mcmv_fds_ts_caixa_202607": "planilha de trabalho social da CAIXA (jul/2026): situação do TS e motivo de atraso; ainda sem prata",
}
inv["descrição"] = inv.tabela.map(DESC).fillna("")
inv.style.format({"linhas": "{:,.0f}"}).hide(axis="index")

schema,tabela,linhas,descrição
bronze,bronze_sftp_canalfgts_entidades,"7,778",
bronze,bronze_sftp_empreendimento_int059,"39,346",
bronze,bronze_sftp_int059_fds_caixa_empreendimentos,916,
bronze,bronze_shpt_fgts_canal_tdom_entidades,"7,421",
bronze,bronze_shpt_mcmv_fds_ts_caixa_202607,"1,009",planilha de trabalho social da CAIXA (jul/2026): situação do TS e motivo de atraso; ainda sem prata
bronze,bronze_shpt_monit_cad_pj_fds_mensal,364,
bronze,bronze_shpt_monit_exec_ts_fds_mensal,141,
bronze,bronze_shpt_monit_mov_financ_fds_mensal,"16,887",
bronze,bronze_shpt_monit_mov_obra_fds_mensal,364,
bronze,bronze_shpt_obra_mensal_fds,"2,783",


## 2. A carteira do Novo MCMV-Entidades: 364 contratos (APFs)

`prata.prata_fds_empreendimento` reúne o cadastro PJ do MONIT, a obra mensal, o INT059 (filtrado em `NOVO PMCMV-E`), o trabalho social e o desembolso acumulado. Uma linha por APF.

In [3]:
emp = cache("fds_empreendimento", "select * from prata.prata_fds_empreendimento")
for c in ["dt_contratacao","dt_inicio_obra","dt_previsao_conclusao","dt_paralisacao","dt_conclusao_obra","dt_entrega","dt_previsao_entrega","dt_ultima_liberacao","dt_aprovacao_pts","dt_substituicao_eo","dt_invasao"]:
    emp[c] = pd.to_datetime(emp[c])
print(f"{len(emp)} APFs | {emp.eo_cnpj.nunique()} entidades organizadoras | {emp.uf.nunique()} UFs | {emp.cod_ibge.nunique()} municípios")
print(f"contratações de {emp.dt_contratacao.min():%d/%m/%Y} a {emp.dt_contratacao.max():%d/%m/%Y} | UH: {emp.quantidade_uh.sum():,.0f} | investimento: R$ {emp.valor_contratado.sum()/1e9:.2f} bi | financiamento FDS: R$ {emp.valor_financiamento_fds.sum()/1e9:.2f} bi".replace(",", "."))
display(pd.crosstab(emp.fase_contrato, emp.situacao_gefus, margins=True))
db.colunas("prata", "prata_fds_empreendimento")

364 APFs | 175 entidades organizadoras | 22 UFs | 226 municípios
contratações de 30/08/2024 a 13/04/2026 | UH: 30.731 | investimento: R$ 3.46 bi | financiamento FDS: R$ 2.63 bi


situacao_gefus,CONCLUIDA,NAO_INICIADA_CLAUSULA_SUSPENSIVA,NORMAL,OBRA_FISICA_CONCLUIDA,PARALISADA,All
fase_contrato,,,,,,
OBRA,0,38,142,0,2,182
PROJETO,1,22,155,2,2,182
All,1,60,297,2,4,364


,column_name,data_type
0,apf,text
1,eo_nome,text
2,eo_cnpj,text
3,co_nivel_hab_eo,integer
4,ic_substituicao_eo,boolean
5,dt_substituicao_eo,date
6,eo_substituta_nome,text
7,eo_substituta_cnpj,text
8,empreendimento_nome,text
9,construtora_nome,text


## 3. As duas fases do contrato: projeto e obra

No Entidades a operação tem dois contratos, cada um com o seu APF: o **contrato de projeto** (aquisição de terreno e elaboração do projeto, modalidades 1 e 2) e o **contrato de obra** (produção das unidades, modalidades 3 e 4). O INT059 traz isso em `fase_contrato`, e a seed `seed_apf_fase_fds` (planilha RELAÇÃO_APF_FASES_FDS do GEFUS) é a fonte de verdade da identidade do empreendimento, consumida por `prata_fds_dim_empreendimento`. Para o Novo MCMV os 364 APFs ainda não estão ligados entre si (cada APF é o seu próprio empreendimento na dimensão): o vínculo projeto → obra só existe para o legado.

In [4]:
dim = cache("fds_dim_empreendimento", "select id_empreendimento, apf, fase_empreendimento, apf_ancora, origem_mapeamento, fase_corrigida from prata.prata_fds_dim_empreendimento")
seed_fase = cache("seed_apf_fase_fds", "select apf::text apf, fase_empreendimento, apf_ancora::text apf_ancora, arquivo_origem from seeds.seed_apf_fase_fds")
print(f"dimensão: {len(dim)} APFs em {dim.id_empreendimento.nunique()} empreendimentos; seed: {len(seed_fase)} APFs em {seed_fase.apf_ancora.nunique()} âncoras")
d_novo = dim[dim.apf.isin(emp.apf)]
print(f"na carteira nova: {len(d_novo)} APFs em {d_novo.id_empreendimento.nunique()} empreendimentos (sem vínculo projeto -> obra)")
t = emp.groupby("fase_contrato").agg(apfs=("apf","size"), uh=("quantidade_uh","sum"), investimento=("valor_contratado","sum"), valor_por_uh_mediano=("valor_por_uh","median"), exec_fisica_media=("percentual_execucao_fisica","mean"), exec_financeira_media=("percentual_execucao_financeira","mean"))
display(t.style.format({"uh": "{:,.0f}", "investimento": "R$ {:,.0f}", "valor_por_uh_mediano": "R$ {:,.0f}", "exec_fisica_media": "{:.1f}%", "exec_financeira_media": "{:.1f}%"}))
display(pd.crosstab(emp.fase_contrato, emp.co_modalidade.map({1: "1 terreno + projeto", 2: "2 projeto", 3: "3 produção de UH", 4: "4 requalificação"})))

dimensão: 1095 APFs em 890 empreendimentos; seed: 1093 APFs em 888 âncoras
na carteira nova: 364 APFs em 364 empreendimentos (sem vínculo projeto -> obra)


,apfs,uh,investimento,valor_por_uh_mediano,exec_fisica_media,exec_financeira_media
fase_contrato,,,,,,
OBRA,182,"16,089","R$ 3,086,485,803","R$ 178,338",8.5%,9.8%
PROJETO,182,"14,642","R$ 370,732,373","R$ 18,598",13.5%,31.9%


co_modalidade,1 terreno + projeto,2 projeto,3 produção de UH,4 requalificação
fase_contrato,,,,
OBRA,3,5,171,3
PROJETO,100,61,19,2


**Leitura.** Metade da carteira (182 APFs) são contratos de projeto, baratos (R$ 24 mil por UH, mediana) e com "execução física" que mede o avanço do projeto, não da obra; a outra metade são contratos de obra (R$ 190 mil por UH). Os 30,8 mil UH da carteira somam as duas fases, então um mesmo empreendimento pode estar contado duas vezes se já passou de projeto a obra. Sem o vínculo entre os dois APFs não dá para medir o tempo entre o contrato de projeto e o contrato de obra, que é uma etapa própria do fluxo do Entidades.

## 4. As entidades organizadoras

`ouro.ouro_fds_panorama_entidade` tem uma linha por CNPJ de EO, com carteira, execução e um score de risco. O nível de habilitação (`co_nivel_hab_eo`, 1 a 6) vem no cadastro PJ; a habilitação em si (SISAD) não está no banco.

In [5]:
pan = cache("fds_panorama_entidade", "select * from ouro.ouro_fds_panorama_entidade")
print(f"{len(pan)} entidades no panorama; {pan.total_empreendimentos.sum()} empreendimentos; {pan.total_uhs_contratadas.sum():,.0f} UH".replace(",", "."))
display(pan.classificacao_risco.value_counts().to_frame("entidades"))
display(pan.sort_values("total_uhs_contratadas", ascending=False).head(10)[["nome_eo","total_empreendimentos","total_uhs_contratadas","qtd_estados_atuacao","qtd_municipios_atuacao","media_execucao_fisica","media_execucao_financeira","qtd_obras_paralisadas","classificacao_risco"]])
display(emp.groupby("co_nivel_hab_eo").agg(entidades=("eo_cnpj","nunique"), apfs=("apf","size"), uh=("quantidade_uh","sum")))

175 entidades no panorama; 364 empreendimentos; 30.731 UH


,entidades
classificacao_risco,
Baixo Risco,114
Médio Risco,59
Alto Risco,2


,nome_eo,total_empreendimentos,total_uhs_contratadas,qtd_estados_atuacao,qtd_municipios_atuacao,media_execucao_fisica,media_execucao_financeira,qtd_obras_paralisadas,classificacao_risco
70,ASS DE MORADORES DO ACAMPAMENTO ESPERANCA DE UM NOVO MILENIO,20,3173,5,10,0.06,0.00,0,Médio Risco
168,CENTRO DE EST ARTC E REF SOBRE ASSENTAMENTOS HUMANOS,10,1402,4,7,10.01,10.53,0,Médio Risco
144,UNIAO POR MORADIA POPULAR DA PARAIBA (UMP-PB),8,999,1,4,18.32,22.86,0,Médio Risco
14,ASSOCIAÇÃO CULTURAL E DE PROMOÇÃO DE CIDADANIA PALMARES EM M,6,996,1,2,20.11,22.38,0,Médio Risco
62,MOVIMENTO DE LUTAS NOS BAIRROS VILAS E FAVELAS,5,783,4,5,0.01,2.56,0,Baixo Risco
49,ASSOC DE APOIO AOS SEM TETO DO ESTADO PE,5,780,1,3,25.98,20.80,0,Médio Risco
98,ASSOC DE APOIO AS CIDADES COMUNID E ASSENT DO SEMI-ARIDO NOR,11,750,1,7,0.45,1.65,0,Médio Risco
61,ASSOCIACAO BENEFICENTE DE DESENVOLVIMENTO DO DISTRITO STELA,12,750,1,5,18.85,33.33,0,Alto Risco
173,ASSOCIACAO DOS MORADORES DO BAIRRO SANTA ROSA,5,739,2,5,24.87,26.91,0,Baixo Risco
129,CENTRO DE PESQUISA FORMACAO E DESENVOLVIMENTO FEMINISTA,3,568,1,2,19.09,16.74,0,Médio Risco


,entidades,apfs,uh
co_nivel_hab_eo,,,
1,23,126,14554
2,19,49,4467
3,16,41,3209
4,27,40,3132
5,25,38,2112
6,65,70,3257


## 5. A série no tempo: onde está o histórico

O GFAR manda retratos (o cadastro, a obra e o TS têm só o mês mais recente); a série está em três lugares: o livro-razão das liberações (`prata_fds_financeiro_mensal`, desde 2011, legado e novo), o eixo histórico `prata_hist_fds_empreendimento` (INT059 mensal desde dezembro de 2019 para o legado; Dados Prioritários mensais de junho de 2024 a março de 2026; obra mensal de 2026) e a evolução financeira por mês.

In [6]:
hr = cache("fds_hist_resumo", """
select fonte_serie, fonte_tabela, count(*) n, count(distinct apf) apfs, min(dt_referencia) ini, max(dt_referencia) fim, count(distinct dt_referencia) meses,
       count(distinct apf) filter (where apf in (select apf from prata.prata_fds_empreendimento)) apfs_carteira_nova,
       count(*) filter (where percentual_execucao_fisica is not null or pc_obra_realizada is not null) linhas_com_fisico,
       count(*) filter (where motivo_paralisacao is not null) linhas_com_motivo_paralisacao
from prata.prata_hist_fds_empreendimento group by 1, 2 order by 1""")
display(hr)
fr = cache("fds_financeiro_resumo", """
select count(*) liberacoes, count(distinct apf) apfs, min(dt_liberacao) ini, max(dt_liberacao) fim, sum(abs(vr_liberado)) filter (where ic_credito='0')/1e9 liberado_bi,
       count(*) filter (where right(apf,6) in (select left(apf,6) from prata.prata_fds_empreendimento)) liberacoes_carteira_nova
from prata.prata_fds_financeiro_mensal""")
display(fr)
print("Cobertura mensal dos Dados Prioritários (fonte snh) para a carteira nova:")
cob = cache("fds_hist_cobertura_novo", """
select dt_referencia, fonte_serie, count(*) apfs, count(*) filter (where coalesce(percentual_execucao_fisica, pc_obra_realizada) > 0) apfs_com_fisico
from prata.prata_hist_fds_empreendimento where apf in (select apf from prata.prata_fds_empreendimento) group by 1, 2 order by 1, 2""")
display(cob.pivot(index="dt_referencia", columns="fonte_serie", values=["apfs","apfs_com_fisico"]).fillna(0).astype(int).tail(12))

,fonte_serie,fonte_tabela,n,apfs,ini,fim,meses,apfs_carteira_nova,linhas_com_fisico,linhas_com_motivo_paralisacao
0,obra_mensal,sharepoint:MONIT_MOV_OBRA,1087,364,2026-04-01,2026-07-01,3,364,1087,0
1,sftp,INT059_MinisterioCidades_FDS_CAIXA_EMPREENDIMENTOS,30761,938,2019-12-01,2026-06-01,68,364,30761,296
2,snh,SNH_dados_prioritarios_af_caixa,14221,917,2024-06-01,2026-03-01,22,357,14193,3104


,liberacoes,apfs,ini,fim,liberado_bi,liberacoes_carteira_nova
0,16887,839,2011-12-29,2026-07-30,4.978488,828


Cobertura mensal dos Dados Prioritários (fonte snh) para a carteira nova:


apfs           apfs_com_fisico          
fonte_serie   obra_mensal sftp  snh     obra_mensal sftp  snh
dt_referencia                                                
2025-08-01              0    0  212               0    0   32
2025-09-01              0    0  247               0    0   37
2025-10-01              0    0  287               0    0   52
2025-11-01              0    0  295               0    0   69
2025-12-01              0    0  334               0    0   85
2026-01-01              0    0  337               0    0   97
2026-02-01              0    0  346               0    0  103
2026-03-01              0    0  357               0    0  115
2026-04-01            361    0    0             129    0    0
2026-05-01            362    0    0             141    0    0
2026-06-01              0  364    0               0  164    0
2026-07-01            364    0    0             184    0    0

## 6. O legado PMCMV-E (2009 a 2019)

O INT059 traz 574 APFs do PMCMV-E antigo com retrato mensal desde dezembro de 2019: situação, físico, desembolso, datas de contratação, conclusão e entrega. É a base de obras concluídas, paralisadas e canceladas com que se mede quanto tempo um contrato do Entidades leva de verdade (notebook 03) e onde ele para.

In [7]:
leg = cache("fds_legado_ultimo", """
select distinct on (apf) apf, nome_empreendimento, uf, municipio, quantidade_uh, quantidade_uh_entregues, status_operacional, situacao_canonica,
       dt_contratacao, dt_inicio_obra, dt_conclusao_obra, dt_entrega_uh, dt_paralisacao, percentual_execucao_fisica, valor_contratado, valor_desembolsado, motivo_paralisacao, dt_referencia
from prata.prata_hist_fds_empreendimento where fonte_serie = 'sftp' and apf not in (select apf from prata.prata_fds_empreendimento)
order by apf, dt_referencia desc""")
for c in ["dt_contratacao","dt_inicio_obra","dt_conclusao_obra","dt_entrega_uh","dt_paralisacao","dt_referencia"]: leg[c] = pd.to_datetime(leg[c])
print(f"{len(leg)} APFs do legado | contratos de {leg.dt_contratacao.min():%Y} a {leg.dt_contratacao.max():%Y} | UH: {leg.quantidade_uh.sum():,.0f} | entregues: {leg.quantidade_uh_entregues.sum():,.0f}".replace(",", "."))
display(leg.groupby("status_operacional").agg(apfs=("apf","size"), uh=("quantidade_uh","sum"), entregues=("quantidade_uh_entregues","sum"), exec_media=("percentual_execucao_fisica","mean"), com_dt_conclusao=("dt_conclusao_obra","count"), com_dt_entrega=("dt_entrega_uh","count")).round(1))
display(leg.groupby(leg.dt_contratacao.dt.year).agg(apfs=("apf","size"), uh=("quantidade_uh","sum"), entregues=("quantidade_uh_entregues","sum")).T)

574 APFs do legado | contratos de 2009 a 2026 | UH: 85.835 | entregues: 37.853


,apfs,uh,entregues,exec_media,com_dt_conclusao,com_dt_entrega
status_operacional,,,,,,
CANCELADO,26,5369,0.0,4.2,4,0
CONCLUIDA,365,43484,30849.0,100.0,199,332
NAO_INICIADA_CLAUSULA_SUSPENSIVA,1,200,0.0,0.0,0,0
NORMAL,15,3076,0.0,56.3,3,0
OBRA_FISICA_CONCLUIDA,81,15419,5227.0,99.7,19,28
PARALISADA,86,18287,1777.0,77.1,24,12


dt_contratacao,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2023,2024,2025,2026
apfs,4.0,59.0,29.0,31.0,64.0,101.0,60.0,93.0,13.0,91.0,7.0,1.0,13.0,7.0,1.0
uh,309.0,5682.0,2641.0,3835.0,8926.0,16374.0,10937.0,14360.0,3171.0,10733.0,1879.0,648.0,4578.0,1706.0,56.0
entregues,213.0,3441.0,2240.0,1309.0,2615.0,9233.0,3555.0,6517.0,1821.0,5209.0,1700.0,0.0,0.0,0.0,0.0


## 7. Seeds e dicionários úteis

Os códigos do MONIT são os mesmos do FAR, então `seeds.far_situacao_obra` decodifica `co_situacao_operacao` do FDS. Para o trabalho social há `seeds.ts_dominio_situacao` (texto da CAIXA, não os códigos numéricos do MONIT). A seed `portaria_698_2026` é a lista de propostas selecionadas no ciclo 2025/26 do Entidades (393 propostas), ainda sem contratos na carteira.

In [8]:
display(cache("seed_far_situacao_obra", "select co_situacao_obra, situacao_obra from seeds.far_situacao_obra"))
display(db.ler_sql("select * from seeds.ts_dominio_situacao"))
display(db.ler_sql("select * from seeds.dominio_motivo_paralisacao"))
p698 = cache("seed_portaria_698_2026", 'select "IDENTIFICACAO DA PROPOSTA" id_proposta, "UF" uf, "MUNICÍPIO" municipio, "CNPJ ENTIDADE" cnpj_eo, "NOME DO EMPREENDIMENTO" nome, "UH" uh from seeds.portaria_698_2026')
print(f"Portaria 698/2026: {len(p698)} propostas, {p698.uh.sum():,.0f} UH, {p698.cnpj_eo.nunique()} entidades, {p698.uf.nunique()} UFs".replace(",", "."))
print("Situação MONIT (co_situacao_operacao) na carteira nova, decodificada pela seed:")
sit = cache("seed_far_situacao_obra", "select co_situacao_obra, situacao_obra from seeds.far_situacao_obra").astype({"co_situacao_obra": int}).set_index("co_situacao_obra").situacao_obra
display(pd.crosstab(emp.co_situacao_operacao.map(sit), emp.situacao_gefus, margins=True))

,co_situacao_obra,situacao_obra
0,1,NÃO INICIADA
1,2,EM ANDAMENTO
2,3,ATRASADA
3,4,PARALISADA
4,5,CONCLUÍDA
5,6,OCUPADA IRREGULARMENTE
6,7,CONCLUÍDA FÍSICA CONCLUÍDA
7,8,CANCELADA OU DISTRATADA
8,9,CONCLUÍDA NÃO ENTREGUE
9,10,CONCLUÍDA E ENTREGUE SEM PENDÊNCIA


,agente_financeiro,situacao_origem,situacao_padronizada,ic_ativo,ic_critico,ic_equivalencia_a_confirmar,observacao
0,CAIXA,CONCLUIDO,Concluído,false,false,false,TS encerrado com AVF atestada
1,CAIXA,EM ANDAMENTO,Em andamento,true,false,false,Execução adequada aos marcos
2,CAIXA,ATRASADO,Atrasado,true,true,false,PTS aprovado sem início ou relatórios atrasados além do prazo da norma
3,CAIXA,PARALISADO,Paralisado,true,true,false,Impedimento para continuar o TS (avaliado pela centralizadora)
4,CAIXA,NAO INICIADO,Não iniciado,false,false,false,Ainda não há motivo para cobrar entrega pelos marcos temporais
5,CAIXA,NÃO INICIADO,Não iniciado,false,false,false,Ainda não há motivo para cobrar entrega pelos marcos temporais
6,BB,CONCLUÍDO,Concluído,false,false,false,TS encerrado
7,BB,CONCLUÍDO C/ GLOSA PGTO FINAL,Concluído,false,false,true,Concluído com glosa no pagamento final
8,BB,EM ANDAMENTO,Em andamento,true,false,false,Execução em curso
9,BB,REGULARIZADO,Em andamento,true,false,true,Pendência resolvida e execução retomada


,valor_bruto,motivo_paralisacao,origem
0,A ENTREGAR,a_entregar,snh
1,EM ANDAMENTO,em_andamento,snh
2,A RETOMAR,a_retomar,snh
3,A RETOMAR - AO/MCI/ASSINATURA,a_retomar_ao_mci_assinatura,snh
4,A RETOMAR - APORTE ENTE PÚBLICO,a_retomar_aporte_ente_publico,snh
5,A RETOMAR - APORTE SUPLEMENTAR,a_retomar_aporte_suplementar,snh
6,A RETOMAR - EM ANÁLISE,a_retomar_em_analise,snh
7,A RETOMAR - EM CONSTRUÇÃO DA PROPOSTA,a_retomar_em_construcao_proposta,snh
8,A RETOMAR - SOLUÇÃO LOCAL,a_retomar_solucao_local,snh
9,DESIMOBILIZAÇÃO ASSINADA,desimobilizacao_assinada,snh


Portaria 698/2026: 393 propostas. 35.820 UH. 200 entidades. 24 UFs
Situação MONIT (co_situacao_operacao) na carteira nova, decodificada pela seed:


situacao_gefus,CONCLUIDA,NAO_INICIADA_CLAUSULA_SUSPENSIVA,NORMAL,OBRA_FISICA_CONCLUIDA,PARALISADA,All
co_situacao_operacao,,,,,,
ATRASADA,0,0,27,0,0,27
CONCLUÍDA,1,0,0,2,0,3
EM ANDAMENTO,0,0,29,0,0,29
NORMAL,0,7,111,0,4,122
NÃO INICIADA,0,53,122,0,0,175
OUTRAS,0,0,7,0,0,7
PARALISADA,0,0,1,0,0,1
All,1,60,297,2,4,364


## 8. O que o FAR tem e o Entidades não tem (ainda)

Quatro ausências mudam o que dá para fazer: não há GEHIS para o FDS (o rótulo semanal da CAIXA, com "em atenção" e "risco de paralisação", cobre só o FAR), não há o retrato mensal `prata_far_caixa_serie` para APFs do FDS (os Dados Prioritários do FDS entram só pelo eixo histórico, até março de 2026), não há consolidado de propostas do GFAR (o funil protocolada, enquadrada, selecionada do FAR não existe aqui) e não há base de beneficiários. Em compensação, o Entidades tem no banco uma planilha de trabalho social da CAIXA com motivo de atraso, ainda na bronze.

In [9]:
chk = cache("fds_lacunas", """
select 'GEHIS (bronze_sftp_gehis_andamento_obra)' fonte, count(distinct nu_apf) apfs_fds from bronze.bronze_sftp_gehis_andamento_obra
 where lpad(regexp_replace(nu_apf,'[^0-9]','','g'),8,'0') in (select apf from prata.prata_fds_empreendimento)
union all select 'retrato mensal CAIXA (prata_far_caixa_serie)', count(distinct apf) from prata.prata_far_caixa_serie where apf in (select apf from prata.prata_fds_empreendimento)
union all select 'Dados Prioritários entregas (prata_fds_dados_prioritarios_entregas)', count(distinct apf) from prata.prata_fds_dados_prioritarios_entregas where apf in (select apf from prata.prata_fds_empreendimento)
union all select 'consolidado de propostas GFAR (prata_far_consolidado)', count(distinct apf) from prata.prata_far_consolidado where apf in (select apf from prata.prata_fds_empreendimento)
union all select 'trabalho social CAIXA jul/2026 (bronze)', count(distinct contrato) from bronze.bronze_shpt_mcmv_fds_ts_caixa_202607
 where lpad(regexp_replace(contrato,'[^0-9]','','g'),8,'0') in (select apf from prata.prata_fds_empreendimento)""")
display(chk)
tsb = cache("fds_ts_caixa_202607", """select contrato, municipio, uf, nome_empreendimento, uh, fase_mcmv, situacao_ts, motivo_situacao_ts_atrasado_paralisado, execucao_ts, obra,
       vr_global_ts, vr_desembolsado, natureza_execucao, instrumento_de_planejamento, data_da_contratacao from bronze.bronze_shpt_mcmv_fds_ts_caixa_202607""")
display(pd.crosstab(tsb.fase_mcmv, tsb.situacao_ts, margins=True))

,fonte,apfs_fds
0,trabalho social CAIXA jul/2026 (bronze),362
1,Dados Prioritários entregas (prata_fds_dados_prioritarios_entregas),0
2,consolidado de propostas GFAR (prata_far_consolidado),0
3,retrato mensal CAIXA (prata_far_caixa_serie),0
4,GEHIS (bronze_sftp_gehis_andamento_obra),0


situacao_ts,ATRASADO,CONCLUIDO,EM ANDAMENTO,PARALISADO,All
fase_mcmv,,,,,
Fase III,51,12,300,0,363
Legado I,0,66,2,14,82
Legado II,39,440,38,47,564
All,90,518,340,61,1009


## 9. O que isso define para os notebooks 02 e 03

1. **Carteira e entidades (02):** `prata_fds_empreendimento` (364 APFs) e `ouro_fds_panorama_entidade` (175 EOs) respondem contratação, execução, desembolso e entrega; a fase do contrato (projeto ou obra) separa duas populações que não podem ser somadas sem cuidado.
2. **Série no tempo (02 e 03):** o físico mensal vem do eixo histórico (`prata_hist_fds_empreendimento`: Dados Prioritários até março de 2026, obra mensal e INT059 em 2026) e o financeiro do livro-razão; a curva S por mês de contrato sai dessas duas.
3. **Vida do contrato (03):** o legado do INT059 (574 APFs, 2009 a 2019, com datas de conclusão e entrega) é a única base com contratos que percorreram o ciclo inteiro; a carteira nova ainda não tem obra concluída.
4. **Lacunas:** sem GEHIS, sem funil de propostas, sem beneficiários, sem vínculo projeto → obra no Novo MCMV, e o trabalho social da CAIXA ainda na bronze.